# 10_deep_model_ablation_and_robustness

Run predefined classical feature-set ablations and robustness checks. Expert hotspot annotations are not used as predictors.

In [1]:

from pathlib import Path
import os, json, shutil, zipfile, hashlib, warnings, math, random, glob
from datetime import datetime, timezone
import pandas as pd
import numpy as np

BASE_DIR = Path(os.environ.get("THERMO_BASE_DIR", "/content"))
PROJECT_NAME = "project_thermography_equine"
PROJECT_ROOT = BASE_DIR / PROJECT_NAME
DATA_ROOT = PROJECT_ROOT / "data"
RAW_DATA_DIR = DATA_ROOT / "raw"
SPLIT_DATA_DIR = DATA_ROOT / "dataset_split"
METADATA_DIR = DATA_ROOT / "metadata"
ANNOTATIONS_DIR = DATA_ROOT / "annotations"
OUTPUT_ROOT = PROJECT_ROOT / "outputs"
CONFIG_DIR = OUTPUT_ROOT / "config"
REPORTS_DIR = OUTPUT_ROOT / "reports"
TABLES_DIR = OUTPUT_ROOT / "tables"
FIGURES_DIR = OUTPUT_ROOT / "figures"
MODELS_DIR = OUTPUT_ROOT / "models"
MODEL_SELECTION_DIR = OUTPUT_ROOT / "model_selection"
PROCESSED_DIR = DATA_ROOT / "processed"
CLEAN_IMAGE_DIR = PROCESSED_DIR / "clean_images"
FEATURES_DIR = OUTPUT_ROOT / "features"

for p in [PROJECT_ROOT, DATA_ROOT, RAW_DATA_DIR, SPLIT_DATA_DIR, METADATA_DIR, ANNOTATIONS_DIR,
          OUTPUT_ROOT, CONFIG_DIR, REPORTS_DIR, TABLES_DIR, FIGURES_DIR, MODELS_DIR,
          MODEL_SELECTION_DIR, PROCESSED_DIR, CLEAN_IMAGE_DIR, FEATURES_DIR]:
    p.mkdir(parents=True, exist_ok=True)

RANDOM_SEED = 42
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

SEARCH_ROOTS = [BASE_DIR, Path('/mnt/data')]

def _existing_roots():
    return [p for p in SEARCH_ROOTS if p.exists()]

def _safe_copy(src: Path, dst: Path, overwrite: bool = False):
    dst.parent.mkdir(parents=True, exist_ok=True)
    if src.resolve() == dst.resolve():
        return dst
    if overwrite or not dst.exists():
        shutil.copy2(src, dst)
    return dst

def find_first(patterns, roots=None, exclude_dirs=(".ipynb_checkpoints",)):
    roots = roots or _existing_roots()
    for root in roots:
        for pattern in patterns:
            for p in sorted(root.rglob(pattern)):
                if any(part in exclude_dirs for part in p.parts):
                    continue
                if p.is_file():
                    return p
    return None

def recover_notebook06_outputs():
    """Recover outputs from Notebook 06 if the user uploaded them to /content instead of project outputs/config."""
    targets = {
        "classical_features.csv": ["classical_features.csv", "classical_features*.csv"],
        "classical_feature_metadata.json": ["classical_feature_metadata.json"],
        "classical_feature_summary.csv": ["classical_feature_summary.csv"],
    }
    recovered = {}
    for target, patterns in targets.items():
        dst = CONFIG_DIR / target
        if dst.exists():
            recovered[target] = str(dst)
            continue
        src = find_first(patterns)
        if src is None:
            recovered[target] = None
            continue
        _safe_copy(src, dst)
        if target.endswith('.csv'):
            _safe_copy(src, FEATURES_DIR / target)
        recovered[target] = str(src)
    meta_path = CONFIG_DIR / "classical_feature_metadata.json"
    if meta_path.exists():
        meta = json.loads(meta_path.read_text(encoding="utf-8"))

        if meta.get("clinical_label_source") == "folder_structure":
            meta["clinical_label_source"] = "expert_curated_folder_structure_based_on_clinical_assessment"
            meta["clinical_label_source_note"] = (
                "Folder structure stores labels assigned by veterinary specialists before model development; "
                "folder names are a technical representation, not an independent diagnostic criterion."
            )
            meta_path.write_text(json.dumps(meta, indent=2), encoding="utf-8")
    return recovered

def load_classical_features_and_metadata():
    recovered = recover_notebook06_outputs()
    features_path = CONFIG_DIR / "classical_features.csv"
    metadata_path = CONFIG_DIR / "classical_feature_metadata.json"
    if not features_path.exists() or not metadata_path.exists():
        raise FileNotFoundError(
            "Notebook 06 outputs are missing. Upload classical_features.csv and classical_feature_metadata.json, "
            "or run Notebook 06 before this notebook. Recovery status: " + json.dumps(recovered, indent=2)
        )
    features = pd.read_csv(features_path)
    metadata = json.loads(metadata_path.read_text(encoding="utf-8"))
    feature_columns = metadata.get("feature_columns", [])
    required = {"horse_id", "split", "label_clinical", "label_binary"}
    missing_required = sorted(required - set(features.columns))
    if missing_required:
        raise KeyError("classical_features.csv is missing required columns: " + ", ".join(missing_required))
    missing_features = [c for c in feature_columns if c not in features.columns]
    if missing_features:
        raise KeyError("Feature columns listed in metadata are absent from classical_features.csv: " + ", ".join(missing_features[:20]))
    allowed_splits = {"train", "valid", "test"}
    bad_splits = sorted(set(features["split"].dropna().astype(str)) - allowed_splits)
    if bad_splits:
        raise ValueError("Unexpected split labels: " + ", ".join(bad_splits))
    overlap = []
    for a in allowed_splits:
        for b in allowed_splits:
            if a < b:
                ia = set(features.loc[features["split"] == a, "horse_id"].astype(str))
                ib = set(features.loc[features["split"] == b, "horse_id"].astype(str))
                common = ia & ib
                if common:
                    overlap.append((a, b, len(common), sorted(list(common))[:5]))
    if overlap:
        raise ValueError("Horse-level leakage detected across splits: " + repr(overlap))
    leakage_like = [c for c in feature_columns if any(token in c.lower() for token in ["hotspot", "annotation", "label", "split", "horse_id", "image_name"])]
    if leakage_like:
        raise ValueError("Potential leakage columns found among predictors: " + ", ".join(leakage_like))
    print("Project root:", PROJECT_ROOT)
    print("Config dir:", CONFIG_DIR)
    print("Loaded Notebook 06 feature table:", features.shape)
    print("Clinical label source:", metadata.get("clinical_label_source"))
    display(features.groupby(["split", "label_clinical"]).size().reset_index(name="n"))
    return features, metadata, feature_columns


In [2]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score, average_precision_score, accuracy_score, balanced_accuracy_score, f1_score, confusion_matrix

features, feature_metadata, all_feature_columns = load_classical_features_and_metadata()
print("Healthy with expert hotspot:", int(features.get("healthy_with_expert_hotspot", pd.Series(False, index=features.index)).astype(bool).sum()))


Project root: /content/project_thermography_equine
Config dir: /content/project_thermography_equine/outputs/config
Loaded Notebook 06 feature table: (347, 46)
Clinical label source: expert_classification_from_folder_structure


,split,label_clinical,n
0,test,healthy,40
1,test,pathological,13
2,train,healthy,179
3,train,pathological,63
4,valid,healthy,38
5,valid,pathological,14


Healthy with expert hotspot: 6


In [3]:
from sklearn.metrics import roc_auc_score, average_precision_score, accuracy_score, balanced_accuracy_score, f1_score, confusion_matrix

def get_prob(model, X):
    if hasattr(model, "predict_proba"):
        return model.predict_proba(X)[:, 1]
    scores = model.decision_function(X)
    return 1 / (1 + np.exp(-scores))

def choose_threshold_youden(y_true, y_prob):
    # Validation-only threshold choice: maximizes sensitivity + specificity - 1.
    y_true = np.asarray(y_true).astype(int)
    y_prob = np.asarray(y_prob).astype(float)
    candidates = np.unique(np.r_[0.0, y_prob, 1.0])
    best = {"threshold": 0.5, "youden_j": -np.inf, "balanced_accuracy": -np.inf}
    for thr in candidates:
        pred = (y_prob >= thr).astype(int)
        tn, fp, fn, tp = confusion_matrix(y_true, pred, labels=[0, 1]).ravel()
        sens = tp / (tp + fn) if (tp + fn) else 0.0
        spec = tn / (tn + fp) if (tn + fp) else 0.0
        youden = sens + spec - 1
        bal = (sens + spec) / 2
        if (youden, bal) > (best["youden_j"], best["balanced_accuracy"]):
            best = {"threshold": float(thr), "youden_j": float(youden), "balanced_accuracy": float(bal), "sensitivity": float(sens), "specificity": float(spec)}
    return best

def metric_dict(y_true, y_prob, threshold=0.5):
    y_true = np.asarray(y_true).astype(int)
    y_prob = np.asarray(y_prob).astype(float)
    y_pred = (y_prob >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
    return {
        "roc_auc": float(roc_auc_score(y_true, y_prob)) if len(np.unique(y_true)) == 2 else np.nan,
        "average_precision": float(average_precision_score(y_true, y_prob)) if len(np.unique(y_true)) == 2 else np.nan,
        "accuracy": float(accuracy_score(y_true, y_pred)),
        "balanced_accuracy": float(balanced_accuracy_score(y_true, y_pred)),
        "f1": float(f1_score(y_true, y_pred, zero_division=0)),
        "sensitivity": float(tp / (tp + fn)) if (tp + fn) else np.nan,
        "specificity": float(tn / (tn + fp)) if (tn + fp) else np.nan,
        "ppv": float(tp / (tp + fp)) if (tp + fp) else np.nan,
        "npv": float(tn / (tn + fn)) if (tn + fn) else np.nan,
        "tn": int(tn), "fp": int(fp), "fn": int(fn), "tp": int(tp),
    }


In [4]:
def feature_set_columns(name):
    if name == "all_features":
        return all_feature_columns
    if name == "thermal_distribution":
        prefixes = ("img_", "top10", "top30", "r_", "g_", "b_")
        return [c for c in all_feature_columns if c.startswith(prefixes)]
    if name == "asymmetry_only":
        return [c for c in all_feature_columns if c.startswith("lr_asym")]
    if name == "texture_gradient":
        return [c for c in all_feature_columns if c.startswith("grad") or c == "entropy_gray"]
    raise ValueError(name)

feature_sets = ["all_features", "thermal_distribution", "asymmetry_only", "texture_gradient"]

def make_model(kind, seed):
    if kind == "logistic_l2":
        return Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
            ("model", LogisticRegression(max_iter=5000, class_weight="balanced", random_state=seed)),
        ])
    if kind == "random_forest":
        return Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("model", RandomForestClassifier(n_estimators=250, class_weight="balanced", random_state=seed, n_jobs=1, min_samples_leaf=2)),
        ])
    raise ValueError(kind)

train_valid = features[features["split"].isin(["train", "valid"])].copy()
test = features[features["split"] == "test"].copy()
if train_valid.empty or test.empty:
    raise ValueError("Development and test sets are required.")

rows = []
for feature_set in feature_sets:
    cols = feature_set_columns(feature_set)
    if not cols:
        warnings.warn(f"No columns found for feature set {feature_set}; skipping.")
        continue
    for kind in ["logistic_l2", "random_forest"]:
        for seed in [1, 7, 42, 101, 2026]:
            model = make_model(kind, seed)
            model.fit(train_valid[cols], train_valid["label_binary"].astype(int).values)
            prob = get_prob(model, test[cols])
            md = metric_dict(test["label_binary"].astype(int).values, prob, threshold=0.5)
            rows.append({"feature_set": feature_set, "model_name": kind, "seed": seed, "n_features": len(cols), "test_subset": "all_test", **md})
            if "healthy_with_expert_hotspot" in test.columns:
                sens = test[~test["healthy_with_expert_hotspot"].astype(bool)].copy()
                if len(sens) and sens["label_binary"].nunique() == 2:
                    prob_s = get_prob(model, sens[cols])
                    md_s = metric_dict(sens["label_binary"].astype(int).values, prob_s, threshold=0.5)
                    rows.append({"feature_set": feature_set, "model_name": kind, "seed": seed, "n_features": len(cols), "test_subset": "excluding_healthy_with_expert_hotspot", **md_s})

ablation_results = pd.DataFrame(rows)
display(ablation_results.sort_values(["test_subset", "roc_auc"], ascending=[True, False]).head(20))


,feature_set,model_name,seed,n_features,test_subset,roc_auc,average_precision,accuracy,balanced_accuracy,f1,sensitivity,specificity,ppv,npv,tn,fp,fn,tp
20,thermal_distribution,logistic_l2,1,25,all_test,0.921154,0.796647,0.830189,0.861538,0.727273,0.923077,0.800,0.600000,0.969697,32,8,1,12
22,thermal_distribution,logistic_l2,7,25,all_test,0.921154,0.796647,0.830189,0.861538,0.727273,0.923077,0.800,0.600000,0.969697,32,8,1,12
24,thermal_distribution,logistic_l2,42,25,all_test,0.921154,0.796647,0.830189,0.861538,0.727273,0.923077,0.800,0.600000,0.969697,32,8,1,12
26,thermal_distribution,logistic_l2,101,25,all_test,0.921154,0.796647,0.830189,0.861538,0.727273,0.923077,0.800,0.600000,0.969697,32,8,1,12
28,thermal_distribution,logistic_l2,2026,25,all_test,0.921154,0.796647,0.830189,0.861538,0.727273,0.923077,0.800,0.600000,0.969697,32,8,1,12
0,all_features,logistic_l2,1,35,all_test,0.892308,0.773189,0.830189,0.861538,0.727273,0.923077,0.800,0.600000,0.969697,32,8,1,12
2,all_features,logistic_l2,7,35,all_test,0.892308,0.773189,0.830189,0.861538,0.727273,0.923077,0.800,0.600000,0.969697,32,8,1,12
4,all_features,logistic_l2,42,35,all_test,0.892308,0.773189,0.830189,0.861538,0.727273,0.923077,0.800,0.600000,0.969697,32,8,1,12
6,all_features,logistic_l2,101,35,all_test,0.892308,0.773189,0.830189,0.861538,0.727273,0.923077,0.800,0.600000,0.969697,32,8,1,12
8,all_features,logistic_l2,2026,35,all_test,0.892308,0.773189,0.830189,0.861538,0.727273,0.923077,0.800,0.600000,0.969697,32,8,1,12


In [5]:
ablation_summary = ablation_results.groupby(["feature_set", "model_name", "test_subset"]).agg(
    n_runs=("seed", "count"),
    roc_auc_mean=("roc_auc", "mean"),
    roc_auc_sd=("roc_auc", "std"),
    balanced_accuracy_mean=("balanced_accuracy", "mean"),
    balanced_accuracy_sd=("balanced_accuracy", "std"),
    f1_mean=("f1", "mean"),
    f1_sd=("f1", "std"),
).reset_index()

ablation_results.to_csv(CONFIG_DIR / "model_ablation_results.csv", index=False)
ablation_results.to_csv(REPORTS_DIR / "model_ablation_results.csv", index=False)
ablation_summary.to_csv(CONFIG_DIR / "model_ablation_summary.csv", index=False)
ablation_summary.to_csv(REPORTS_DIR / "model_ablation_summary.csv", index=False)
ablation_summary.to_csv(TABLES_DIR / "table_model_ablation_summary.csv", index=False)

methods_text = (
    "Robustness analyses evaluated predefined classical feature subsets and random seeds using the locked development "
    "set for model fitting and the independent test set for evaluation. A sensitivity analysis repeated test-set evaluation "
    "after excluding clinically healthy images with expert-marked hotspots; these images remained clinically healthy and "
    "were not relabeled. Expert hotspot annotations were not used as classification predictors."
)
(CONFIG_DIR / "methods_ablation_robustness_text.txt").write_text(methods_text, encoding="utf-8")
(REPORTS_DIR / "methods_ablation_robustness_text.txt").write_text(methods_text, encoding="utf-8")
print(methods_text)
display(ablation_summary)


Robustness analyses evaluated predefined classical feature subsets and random seeds using the locked development set for model fitting and the independent test set for evaluation. A sensitivity analysis repeated test-set evaluation after excluding clinically healthy images with expert-marked hotspots; these images remained clinically healthy and were not relabeled. Expert hotspot annotations were not used as classification predictors.


,feature_set,model_name,test_subset,n_runs,roc_auc_mean,roc_auc_sd,balanced_accuracy_mean,balanced_accuracy_sd,f1_mean,f1_sd
0,all_features,logistic_l2,all_test,5,0.892308,0.000000,0.861538,0.000000,0.727273,0.000000
1,all_features,logistic_l2,excluding_healthy_with_expert_hotspot,5,0.909502,0.000000,0.873303,0.000000,0.774194,0.000000
2,all_features,random_forest,all_test,5,0.793462,0.011979,0.588462,0.034401,0.347100,0.069544
3,all_features,random_forest,excluding_healthy_with_expert_hotspot,5,0.799095,0.008374,0.579638,0.034401,0.347100,0.069544
4,asymmetry_only,logistic_l2,all_test,5,0.546154,0.000000,0.545192,0.000000,0.380952,0.000000
5,asymmetry_only,logistic_l2,excluding_healthy_with_expert_hotspot,5,0.576923,0.000000,0.557692,0.000000,0.421053,0.000000
6,asymmetry_only,random_forest,all_test,5,0.476154,0.033158,0.522115,0.017201,0.217143,0.038333
7,asymmetry_only,random_forest,excluding_healthy_with_expert_hotspot,5,0.488235,0.034955,0.525792,0.017201,0.228421,0.040014
8,texture_gradient,logistic_l2,all_test,5,0.673077,0.000000,0.621154,0.000000,0.450000,0.000000
9,texture_gradient,logistic_l2,excluding_healthy_with_expert_hotspot,5,0.710407,0.000000,0.654977,0.000000,0.514286,0.000000
